# 01 — Iterators, Generators and Itertools

**Level:** Intermediate  ·  **Language:** Python 3.14  ·  **Series:** Python/Intermediate

this notebook Python of **lazy evaluation** engine then kholta is. Jab tum `for x in something:` likhte is,
Python inside inside jo protocol chalata is — wahi is notebook of mauzoo is.

## This notebook me what seekhenge

- **Iterable vs Iterator** — dono me farq what is (interview of favourite sawal)
- `iter()` / `next()` and `StopIteration` of asli role
- `for` loop of inside of **desugared** form (manual loop likh kar dekhenge)
- **Custom iterator class** — `__iter__` / `__next__` khud implement to do
- **Generator functions** and `yield` — 3 line me iterator
- **Generator expressions** — comprehension of lazy bhai
- **Lazy evaluation** of faida: `sys.getsizeof` + timing from memory/speed comparison
- **Infinite generators** and unko safely kaatna (`islice`)
- `yield from` — delegation
- Generator of advanced controls: `send()`, `throw()`, `close()` (coroutine of jhalak)
- **Pipeline pattern** — Unix pipes like data processing (log analysis)
- **itertools** of poora tour: `count, cycle, repeat, chain, islice, tee, zip_longest, product,
  permutations, comwithouttions, groupby, accumulate, starmap, compress, dropwhile, takewhile,
  filterfalse, pairwise, batched`
- Ek "bara file" `io.StringIO` from **lazily** parhna (memory-safe file processing)
- Gotchas, Real-world problems, **LeetCode problems** (341, 284, 173, 1291), Interview Q&A, Practice, Cheat-sheet

## Prerequisites

- Python basics: `for` loop, `list`/`dict`/`set`, function define to do
- List comprehension of basic idea
- Classes of thora sa taaruf (`__init__`, `self`) — NB me phir bhi samjha denge

> **Tip:** Cells then above from neeche chalao. Har cell agle cell of variables but depend kar can is.

---
## 1. Iterable vs Iterator — bunyadi farq

this do lafz log got dete are. Farq crystal clear kar lo:

| | **Iterable** | **Iterator** |
|---|---|---|
| what is | Jis but loop chal can is | Wo cheez jo **agli value** deti is |
| Zaroori method | `__iter__()` | `__iter__()` **and** `__next__()` |
| Misaal | `list`, `tuple`, `str`, `dict`, `set`, `range` | `list_iterator`, generator, `zip` object, `map` object |
| State rakhta is? | not (position yaad not) | **yes** — kahan tak pohancha happened is |
| Dobara loop? | yes, har baar new iterator banta is | **not** — ek baar khatam then khatam |

**Rishta:** `iterable` from `iter()` call kar of **iterator** nikalta is.

```
iterable  --iter()-->  iterator  --next()-->  value, value, value, ... StopIteration
```

**why important?** because:
1. `for` loop isi protocol but runs is.
2. Lazy evaluation (memory bachana) isi of wajah from mumkin is.
3. Interview me "list and generator me farq" of jawab yahin from aata is.

In [ ]:
# ---- Iterable vs Iterator practically dekhte are ----

nums = [10, 20, 30]          # this ek ITERABLE is (list)

# 1) what list of paas __iter__ is?
print("list me __iter__ is?  ", hasattr(nums, "__iter__"))
# 2) what list of paas __next__ is? -> not. Is for list iterator not is.
print("list me __next__ is?  ", hasattr(nums, "__next__"))

print("-" * 45)

# iter() lagao -> ITERATOR milta is
it = iter(nums)
print("iter(nums) of type:    ", type(it).__name__)   # list_iterator
print("iterator me __iter__?  ", hasattr(it, "__iter__"))  # True (iterator bhi iterable is is)
print("iterator me __next__?  ", hasattr(it, "__next__"))  # True <- yahi asli farq is

### Output of matlab

- `list` of paas sirf `__iter__` is → wo **iterable** is, iterator not.
- `iter(nums)` ne `list_iterator` object banew jiske paas **dono** methods are.
- **Har iterator khud bhi iterable is is** (`__iter__` return does is `self`) — is for tum iterator but
  bhi `for` chala can is. Magar ulta not: har iterable iterator not is.

**Interview tip:** Ek line of jawab —
*"Iterable wo is jis from iterator banew ja can is; iterator wo is jo `__next__` from values gives is and apni
position of state rakhta is."*

---
## 2. `next()` and `StopIteration`

`next(iterator)` iterator then "ek qadam" chalata is. Jab values khatam is jayein then Python
`StopIteration` **exception** phenkta is — this error not, ek **signal** is of "bas, khatam".

**Syntax:**
```python
next(iterator)              # khatam hone but StopIteration raise karega
next(iterator, default)     # khatam hone but exception of bajaye default return karega
```

In [ ]:
# ---- next() ek ek qadam ----
it = iter([1, 2, 3])          # 3 values that iterator

print(next(it))               # 1
print(next(it))               # 2
print(next(it))               # 3

# Ab iterator khali is. Agla next() StopIteration dega.
# Jaan boojh kar error dikhana is -> try/except me wrap do are (notebook crash no is)
try:
    next(it)
except StopIteration:
    print("StopIteration aa gaya -> iterator exhausted (khatam) is chuka is")

# Safe tareeqa: default value do, exception hi not aayega
print("default of sath:", next(it, "KHATAM"))

In [ ]:
# ---- Iterator ONE-SHOT is is: this gotcha bohat log then pakadta is ----

nums = [1, 2, 3]

it = iter(nums)
print("pehla  loop:", [x for x in it])   # sab values consume is gayin
print("dusra  loop:", [x for x in it])   # KHALI list! iterator dobara not runs

print("-" * 45)

# but ITERABLE (list) but dobara loop exactly runs is,
# because har `for` new iterator banata is:
print("list but pehla loop:", [x for x in nums])
print("list but dusra loop:", [x for x in nums])

### Output of matlab / Gotcha

Dusri list **khali** aayi because `it` apni position aakhir tak le ja chuka tha.
Iterator me "rewind" of koi tareeqa not is — new iterator banana padta is.

**Yahan log phaste are:** `zip()`, `map()`, `filter()`, `enumerate()`, and `reversed()` sab **iterators**
return do are (Python 3 me). Un but do baar loop chalao then dusri baar khali milega.

In [ ]:
# ---- Real gotcha: zip object do baar use to do ----
names  = ["Ali", "Sara", "Bilal"]
marks  = [90, 85, 78]

pairs = zip(names, marks)        # this ek ITERATOR is, list not

print("count to do of koshish:", len(list(pairs)))   # 3 -> magar isne pairs consume kar liya!
print("ab loop do         :", list(pairs))         # [] <- khali, because khatam is gaya

print("-" * 45)
# HAL: if dobara should then list me materialize kar lo
pairs = list(zip(names, marks))  # ab this list is (iterable, reusable)
print("length:", len(pairs))
print("loop  :", pairs)

---
## 3. `for` loop inside from kaisa dikhta is?

`for` loop koi jadoo not. this exactly isi tarah "desugar" is is:

```python
for x in iterable:          #  ==  it = iter(iterable)
    body(x)                 #      while True:
                            #          try:    x = next(it)
                            #          except StopIteration: break
                            #          body(x)
```

Chalo isko haath from likh kar sabit do are.

In [ ]:
# ---- `for` loop then manually implement to do ----

def manual_for(iterable, action):
    # Step 1: iterable from iterator create
    it = iter(iterable)
    while True:
        try:
            item = next(it)          # Step 2: agli value maango
        except StopIteration:
            break                    # Step 3: signal got -> loop tordo
        action(item)                 # Step 4: body chalao

# Test: normal for loop like hi behaviour
print("for loop:   ", end="")
for ch in "PYTHON":
    print(ch, end=" ")
print()

print("manual_for: ", end="")
manual_for("PYTHON", lambda ch: print(ch, end=" "))
print()

**Interview tip:** if poocha jaye *"`for` loop internally how kaam does is?"* — bas yahi 4 steps bata do:
`iter()` → `next()` loop → `StopIteration` but break. Is ek jawab from pata chal jata is of tumhe protocol samajh is.

---
## 4. Custom Iterator Class — `__iter__` / `__next__`

if tum apni class then `for` loop me chalana chahte is, then **iterator protocol** implement do:

- `__iter__(self)` → iterator object return kare (if khud class hi iterator is then `return self`)
- `__next__(self)` → agli value return kare, ya `raise StopIteration` kare jab khatam is

**Kahan use is is?** Jab data then step-by-step generate to do is — paging API, sensor readings,
custom range, tree traversal.

In [ ]:
# ---- Custom iterator: Countdown (N from 1 tak ulta ginna) ----

class Countdown:
    # __init__: sirf setup, koi value generate not
    def __init__(self, start):
        self.start = start           # asal starting value (yaad rakhi, badalte not)

    def __iter__(self):
        # NOTE: har baar new state banate are -> is for Countdown REUSABLE iterable is
        self.current = self.start
        return self                  # khud iterator is, is for self

    def __next__(self):
        if self.current <= 0:        # khatam hone of shart
            raise StopIteration      # signal: bas is gaya
        value = self.current         # jo return to do is
        self.current -= 1            # state ahead (yahan behind) barhao
        return value


print("for loop from:", list(Countdown(5)))
print("dobara     :", list(Countdown(5)))    # reusable, because __iter__ state reset does is

# Manually bhi chala can is
c = Countdown(3)
it = iter(c)
print("manual:", next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration -> countdown khatam")

In [ ]:
# ---- Aik zyada realistic misaal: paged API readelay (fake data, koi network not) ----

class PagedResults:
    # Sochlo API ek time me sirf `page_size` records gives is.
    # Iterator user then ek-ek record gives is, paging inside chupa gives is.
    def __init__(self, records, page_size=3):
        self._records = records
        self._page_size = page_size

    def __iter__(self):
        self._i = 0                              # kis record but are
        self._pages_fetched = 0                  # kitni "API calls" happened
        return self

    def __next__(self):
        if self._i >= len(self._records):
            raise StopIteration
        # new page shuru is raha is? then "fetch" of message dikhao
        if self._i % self._page_size == 0:
            self._pages_fetched += 1
            print(f"   [API] page {self._pages_fetched} fetch of gayi...")
        rec = self._records[self._i]
        self._i += 1
        return rec


users = [f"user{n}" for n in range(1, 8)]        # 7 records
print("Sab users ghoomte are (paging inside chup happened is):")
for u in PagedResults(users, page_size=3):
    print("   ->", u)

### Output of matlab

`[API] page N fetch of gayi...` sirf tab print happened jab new page should tha. User of code me koi
paging logic not — wo simply `for u in PagedResults(...)` likhta is. **Yehi iterator of asli faida is:
complexity chhupa then take.**

### Common mistakes (custom iterator)

1. `__iter__` me `return self` bhool jana → `Typeerror: iter() returned non-iterator`.
2. `__next__` me `StopIteration` raise no to do → **infinite loop**.
3. State `__init__` me rakhna and `__iter__` me reset no to do → class sirf **ek baar** loop hogi
   (iterator ban jayegi, reusable iterable not).

In [ ]:
# ---- Mistake #1 and #3 of live demo (try/except me safe) ----

class BadIter:
    def __iter__(self):
        return 42                 # GALAT: iterator not, int return done

try:
    for x in BadIter():
        pass
except Typeerror as e:
    print("Mistake #1 ->", type(e).__name__, ":", e)

print("-" * 45)

class OneShotRange:
    # State __init__ me is and __iter__ reset not does -> sirf ek baar chalega
    def __init__(self, n):
        self.i = 0
        self.n = n
    def __iter__(self):
        return self               # koi reset not
    def __next__(self):
        if self.i >= self.n:
            raise StopIteration
        self.i += 1
        return self.i

r = OneShotRange(4)
print("Mistake #3 -> pehla loop:", list(r))
print("Mistake #3 -> dusra loop:", list(r), "  <- khali! state reset not happened")

---
## 5. Generator functions and `yield`

above that class 10-12 lines of thi. **Generator** from wahi kaam 3 line me:

- Function of inside `return` of jagah **`yield`** write.
- Aisa function call to do but body **chalti not** — ek **generator object** (iterator) milta is.
- Har `next()` but body `yield` tak chalti is, value deti is, and **wahin ruk jati is** (state mehfooz).
- Function khatam hone but Python khud `StopIteration` raise kar gives is.

**Syntax:**
```python
def gen(...):
    yield value1
    yield value2
```

**why use do?** Kam code, kam memory, lazy evaluation, and infinite sequences mumkin.

In [ ]:
# ---- Class vs Generator: wahi Countdown, 3 lines me ----

def countdown(start):
    n = start
    while n > 0:
        yield n            # value do and YAHIN ruk jao
        n -= 1             # agle next() but yahan from chalna shuru hoga

print("generator from:", list(countdown(5)))

g = countdown(3)
print("type:", type(g).__name__)                  # generator
print("iterable?", hasattr(g, "__iter__"), " iterator?", hasattr(g, "__next__"))
print("values:", next(g), next(g), next(g))
try:
    next(g)
except StopIteration:
    print("StopIteration -> generator function of body khatam")

In [ ]:
# ---- `yield` but execution RUK jati is: isko print from sabit do are ----

def noisy():
    print("   [gen] shuru happened")
    yield 1
    print("   [gen] first yield of after chala")
    yield 2
    print("   [gen] doosre yield of after chala")
    print("   [gen] body khatam -> StopIteration ayega")

g = noisy()
print("generator object ban gaya -- gaur do: abhi tak body of koi print not happened")
print("next() #1 ->", next(g))
print("next() #2 ->", next(g))
try:
    next(g)
except StopIteration:
    print("next() #3 -> StopIteration")

### Output of matlab (bohat ahem)

Gaur do of `g = noisy()` but `[gen] shuru happened` print **not** happened! Generator of body **first `next()`**
but chalna shuru is is. this "lazy" hone of pehla saboot is.

**Interview tip:** *"Generator function call to do but what is is?"* → *"Body not chalti; ek generator
object (iterator) return is is. Body first `next()` but shuru is is and har `yield` but suspend is
jati is, apna local state mehfooz rakhte happened."*

In [ ]:
# ---- Generator me `return` of matlab: StopIteration of value ----

def with_return():
    yield "a"
    yield "b"
    return "MAIN RETURN VALUE"      # generator me return = "bas khatam", value StopIteration me jati is
    # yahan of after some not chalega

g = with_return()
print(next(g))
print(next(g))
try:
    next(g)
except StopIteration as e:
    # return of value e.value me gets is (normally ignore is is)
    print("StopIteration of value:", e.value)

# and dhyan do: `for` loop return value then ignore kar gives is
print("for loop:", list(with_return()))

---
## 6. Generator Expressions

List comprehension of lazy version. Sirf brackets badalte are:

```python
[x * x for x in range(5)]     # list comprehension  -> poori list foran banti is
(x * x for x in range(5))     # generator expression -> lazy, some not banta
```

if function call of **sirf ek** argument is then bracket bhi not should:
```python
sum(x * x for x in range(5))  # clean and memory-friendly
```

In [ ]:
# ---- listcomp vs genexp ----
listcomp = [x * x for x in range(6)]        # foran calculate
genexp   = (x * x for x in range(6))        # lazy, sirf recipe

print("listcomp:", listcomp, type(listcomp).__name__)
print("genexp  :", genexp, type(genexp).__name__)   # object dikhata is, values not

print("genexp then materialize:", list(genexp))
print("dobara materialize   :", list(genexp), " <- khali, one-shot tha")

print("-" * 45)
# Aggregation functions of inside bracket of zaroorat not
print("sum   :", sum(x * x for x in range(6)))
print("max   :", max(len(w) for w in ["Pakistan", "Lahore", "Karachi"]))
print("any   :", any(n % 7 == 0 for n in [3, 5, 14, 9]))
print("all   :", all(n > 0 for n in [3, 5, 14, 9]))

### Kab kaun sa?

| Situation | Use |
|---|---|
| Values dobara should / index should / `len()` should | **list comprehension** |
| Sirf ek baar loop to do is (`sum`, `max`, `any`, `for`) | **generator expression** |
| Data bara is / infinite is | **generator expression** (lazzat memory me) |
| Chhota data + readability | list comprehension theek is |

**Gotcha:** `len(genexp)` kaam not does (`Typeerror`), and `genexp[0]` bhi not (subscript not is).

In [ ]:
# ---- Genexp of do gotchas ----
g = (x for x in range(5))

try:
    print(len(g))
except Typeerror as e:
    print("len(genexp) ->", type(e).__name__, ":", e)

try:
    print(g[0])
except Typeerror as e:
    print("genexp[0]   ->", type(e).__name__, ":", e)

# Hal: itertools.islice from "index like" kaam, ya list() bana lo
import itertools
g2 = (x for x in range(5))
print("islice from 3rd value:", list(itertools.islice(g2, 2, 3)))

---
## 7. Lazy Evaluation — memory and speed of asli faida

**Lazy** of matlab: value **sirf tab** create jab maangi jaye. Iske do bare faide:

1. **Memory** — poora data RAM me not aata (constant memory).
2. **Time-then-first-result** — pehli value foran get jati is, poora dataset process hone of infastar not.

Chalo `sys.getsizeof` and `time.perf_counter` from numbers dekh lein.

In [ ]:
import sys
import time

N = 1_000_000

# ---- 1) MEMORY comparison ----
big_list = [x * x for x in range(N)]          # 1 million ints of poori list
gen_obj  = (x * x for x in range(N))          # sirf recipe

list_bytes = sys.getsizeof(big_list)
gen_bytes  = sys.getsizeof(gen_obj)

print(f"list  ({N:,} items) : {list_bytes:>10,} bytes  (~{list_bytes/1024/1024:.2f} MB)")
print(f"generator            : {gen_bytes:>10,} bytes  (~{gen_bytes} bytes, constant!)")
print(f"farq                 : ~{list_bytes // gen_bytes:,}x chhota")

# NOTE: getsizeof list of inside those int objects then count not does,
# yani asli farq isse bhi ZYADA is. Neeche wo bhi naap lete are.
deep = list_bytes + sum(sys.getsizeof(v) for v in big_list[:1000]) // 1000 * len(big_list)
print(f"list of approx TOTAL : ~{deep/1024/1024:.1f} MB (ints then count do happened)")

del big_list      # RAM wapis kar do

In [ ]:
# ---- 2) TIME-then-FIRST-RESULT comparison ----
# Kaam: 1 million numbers me from pehla aisa number dhoondo jo 7 and 11 dono from divide is.

N = 1_000_000

# Eager tareeqa: first poori list create, PHIR dhoondo
t0 = time.perf_counter()
all_matches = [x for x in range(N) if x % 7 == 0 and x % 11 == 0 and x > 0]
first_eager = all_matches[0]
t_eager = time.perf_counter() - t0

# Lazy tareeqa: generator + next() -> pehla match get hi ruk jao
t0 = time.perf_counter()
lazy = (x for x in range(N) if x % 7 == 0 and x % 11 == 0 and x > 0)
first_lazy = next(lazy)
t_lazy = time.perf_counter() - t0

print(f"eager: pehla match = {first_eager}, time = {t_eager*1000:.2f} ms  (poori list bani)")
print(f"lazy : pehla match = {first_lazy}, time = {t_lazy*1000:.2f} ms  (77 tak ja kar ruk gaya)")
print(f"speedup (pehli value of for): ~{t_eager/max(t_lazy, 1e-9):.0f}x")
assert first_eager == first_lazy == 77   # dono of jawab same

### Output of matlab

- **Memory:** generator of size **constant** (~200 bytes) is, list of size data of sath barhta is.
- **Time:** lazy version ne sirf 77 tak kaam done, baqi 999,923 numbers then chhua bhi not.

### Kab lazy not should?

- Data chhota is and tumhe **kai baar** ghoomna is → list behtar (generator dobara not chalega).
- `len()`, indexing, `sort()`, `reverse()` should → list.
- Random access should → list.

**Interview tip:** *"Generator kab use karoge?"* → *"Jab data bara/infinite is, ya streaming is, ya sirf ek
baar traverse to do is — memory O(1) rehti is. Jab reuse/indexing should then list."*

---
## 8. Infinite Generators — and unko safely kaatna

Generator kabhi khatam no hone that bhi is can is (`while True: yield ...`). this theek is — bas
**consume do time** limit lagana zaroori is warna program hang is jayega.

Kaatne of tareeqe:
- `itertools.islice(gen, n)` → pehli `n` values
- loop of inside `break`
- `zip(gen, range(n))`
- `itertools.takewhile(pred, gen)`

**Kabhi no do:** `list(infinite_gen)` — this kabhi wapis not aayega (memory khatam).

In [ ]:
import itertools

# ---- Infinite generator #1: natural numbers ----
def naturals(start=1):
    n = start
    while True:          # kabhi khatam not is
        yield n
        n += 1

# Safe consumption: islice from pehli 10
print("pehli 10       :", list(itertools.islice(naturals(), 10)))

# break from manually
squares_found = []
for n in naturals():
    if n * n > 200:
        break                       # 200 from bara square get gaya -> ruk jao
    squares_found.append(n * n)
print("squares <= 200 :", squares_found)

# takewhile from declaratively
print("takewhile      :", list(itertools.takewhile(lambda n: n < 8, naturals())))

In [ ]:
# ---- Infinite generator #2: Fibonacci (classic) ----
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b        # tuple unpacking from ek line me swap+add

print("pehli 12 fib :", list(itertools.islice(fibonacci(), 12)))

# 1000 from big pehli fib number
first_big = next(f for f in fibonacci() if f > 1000)
print("1000 from big pehli fib:", first_big)
assert first_big == 1597

# ---- Infinite generator #3: cyclic ID assigner (round-robin) ----
def round_robin_servers(servers):
    while True:
        for s in servers:
            yield s                 # list khatam are hi phir shuru

rr = round_robin_servers(["srv-A", "srv-B", "srv-C"])
assigned = [next(rr) for _ in range(7)]     # 7 requests
print("round robin  :", assigned)

### Gotcha: infinite generator then galti from materialize to do

```python
list(naturals())        # ❌ program hang / Memoryerror
sum(naturals())         # ❌ same
sorted(naturals())      # ❌ same
max(naturals())         # ❌ same
```

Sab aggregation functions **poora** iterator consume do are. Infinite of sath sirf lazy operations
(`islice`, `takewhile`, `next`, `break`) chalao.

---
## 9. `yield from` — delegation

`yield from other_iterable` of matlab: *"is iterable of saari values meri taraf from yield kar do"*.

```python
# old tareeqa                    # yield from of sath
for x in sub:                       yield from sub
    yield x
```

Faida sirf chhota code not — `yield from` **send/throw/return** bhi properly forward does is
(sub-generator of sath two-way communication chalti rehti is).

**Sab from common use:** recursion — nested structures then flatten to do, tree traverse to do.

In [ ]:
# ---- Basic: manual loop vs yield from ----
def manual_chain(a, b):
    for x in a:
        yield x
    for x in b:
        yield x

def yf_chain(a, b):
    yield from a          # exactly wahi kaam, ek line
    yield from b

print("manual_chain:", list(manual_chain([1, 2], "ab")))
print("yf_chain    :", list(yf_chain([1, 2], "ab")))
assert list(manual_chain([1, 2], "ab")) == list(yf_chain([1, 2], "ab"))

In [ ]:
# ---- Recursion of asli power: nested list flatten (kisi bhi depth tak) ----

def flatten(nested):
    for item in nested:
        # if item khud ek list/tuple is then us but RECURSE do
        if isinstance(item, (list, tuple)):
            yield from flatten(item)      # sub-generator of values pass-through
        else:
            yield item                    # scalar -> seedha give do

data = [1, [2, 3, [4, [5, 6]], 7], 8, (9, [10])]
print("nested :", data)
print("flat   :", list(flatten(data)))
assert list(flatten(data)) == [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Lazy is: pehli 4 values of for poora structure walk not is
print("pehli 4:", list(itertools.islice(flatten(data), 4)))

In [ ]:
# ---- yield from of sath tree traversal (in-ordelay) ----

class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def in_ordelay(node):
    if node is None:
        return                     # generator me plain return = "khatam"
    yield from in_ordelay(node.left)  # first poora left subtree
    yield node.val                  # phir khud
    yield from in_ordelay(node.right) # phir right subtree

#         8
#       /   \
#      3     10
#     / \      \
#    1   6      14
tree = Node(8, Node(3, Node(1), Node(6)), Node(10, None, Node(14)))
print("in-ordelay:", list(in_ordelay(tree)))
assert list(in_ordelay(tree)) == [1, 3, 6, 8, 10, 14]   # BST of in-ordelay = sorted

---
## 10. Generator of advanced controls: `send()`, `throw()`, `close()`

Ab tak humne generator from sirf values **li** are. Magar generator me values **bheji** bhi ja can are —
yahan from "coroutine" of concept shuru is is.

| Method | what does is |
|---|---|
| `gen.send(value)` | `yield` expression of jagah `value` daalta is and generator then ahead chalata is |
| `gen.throw(ExcType)` | Generator of inside, current `yield` of jagah exception raise does is |
| `gen.close()` | Generator of inside `GeneratorExit` raise does is → cleanup of mauqa |
| `gen.gi_frame` / `.gi_running` | Introspection (kahan ruka happened is) |

**Ahem rule:** `send()` from first generator then **prime** to do padta is — yani ek `next(gen)` ya
`gen.send(None)` chalao taake wo first `yield` but pohanch jaye.

In [ ]:
# ---- send(): running average calculator (coroutine style) ----

def running_average():
    total, count, avg = 0.0, 0, None
    while True:
        # `yield avg` do kaam does is: avg outside bhejta is and
        # send() from aayi value `new_value` me le takes is
        new_value = yield avg
        if new_value is None:
            continue                       # None bheja gaya -> ignore
        total += new_value
        count += 1
        avg = total / count

ra = running_average()
print("prime (pehla next):", next(ra))     # None -> generator first yield but ruk gaya

for v in [10, 20, 30, 40]:
    print(f"send({v:>2}) -> running avg = {ra.send(v)}")

ra.close()                                  # kaam khatam

In [ ]:
# ---- send() of gotcha: prime kiye baghair send to do ----
ra2 = running_average()
try:
    ra2.send(10)          # GALAT: abhi generator start hi not happened
except Typeerror as e:
    print("Gotcha ->", type(e).__name__, ":", e)
    print("HAL: first next(gen) ya gen.send(None) do (priming)")

# Sahi tareeqa
ra2.send(None)            # prime
print("prime of after send(10):", ra2.send(10))
ra2.close()

In [ ]:
# ---- close() and GeneratorExit: cleanup guarantee ----

def resource_worker(name):
    print(f"   [open] {name} khul gaya")
    try:
        while True:
            item = yield
            print(f"   [work] {name} ne process done: {item}")
    except GeneratorExit:
        # close() but this runs is -> yahan cleanup do
        print(f"   [close] {name} band is gaya (GeneratorExit)")
        raise                     # dobara raise to do acha practice is
    finally:
        print(f"   [finally] {name} of finally bhi chala")

w = resource_worker("db-conn")
next(w)                # prime -> [open] print hoga
w.send("row-1")
w.send("row-2")
w.close()              # -> GeneratorExit + finally
print("generator band hone of after send to do:")
try:
    w.send("row-3")
except StopIteration:
    print("   StopIteration -> band generator me some not bhej can")

In [ ]:
# ---- throw(): generator of inside exception inject to do ----

def resilient_worker():
    processed, errors = 0, 0
    while True:
        try:
            item = yield (processed, errors)
            processed += 1
        except Valueerror as e:
            # outside from throw() done gaya error yahan pakda gaya
            errors += 1
            print(f"   [gen] Valueerror handle done: {e}")

w = resilient_worker()
next(w)                                      # prime
print("send('a')      ->", w.send("a"))
print("send('b')      ->", w.send("b"))
print("throw(Valueerror):")
print("   result      ->", w.throw(Valueerror("kharab data")))
print("send('c')      ->", w.send("c"), " <- generator zinda is, chal raha is")
w.close()

### Output of matlab

`throw()` of after bhi generator **zinda** raha because usne exception then inside `try/except` me pakad liya.
if generator handle no does, then exception outside (caller tak) aa jata and generator band is jata.

**Interview tip:** *"`send()` of faida?"* → *"Two-way communication. Generator sirf producer not, consumer
bhi ban jata is — yehi `async/await` coroutines of bunyad is. Practically aaj asyncio use do, magar
concept samajhna zaroori is."*

---
## 11. Pipeline Pattern — Unix pipes Python me

this generators of **sab from qeemti** real-world use is. Har generator ek chhota kaam does is and
apna output agle then give gives is:

```
read_lines  ->  filter_errors  ->  parse  ->  add_field  ->  summarize
   (lazy)          (lazy)         (lazy)      (lazy)         (consume)
```

**Faida:** poora data kabhi memory me not aata — chahe file 50 GB of is. Har stage ek time me **ek record**
handle does is. and stages then lego blocks of tarah reuse/reordelay kar can is.

Hum `io.StringIO` from ek fake log file banayenge (koi outside of file not should).

In [ ]:
import io

# ---- Fake log "file" (asli file of tarah line-by-line parhi ja can is) ----
LOG_TEXT = '''\
2026-03-01 10:00:01 INFO  user=ali    action=login    ms=120
2026-03-01 10:00:04 error user=sara   action=payment  ms=980  msg=card_declined
2026-03-01 10:00:07 INFO  user=bilal  action=browse   ms=45
2026-03-01 10:00:09 WARN  user=ali    action=upload   ms=1500
2026-03-01 10:00:12 error user=ali    action=payment  ms=1100 msg=timeout
2026-03-01 10:00:15 INFO  user=sara   action=logout   ms=30
2026-03-01 10:00:19 error user=zain   action=payment  ms=760  msg=card_declined
2026-03-01 10:00:24 INFO  user=zain   action=browse   ms=60
'''

def open_fake_log():
    # io.StringIO ek "file like" object gives is -> for line in f: runs is
    return io.StringIO(LOG_TEXT)

with open_fake_log() as f:
    print("Log of pehli 3 lines:")
    for line in itertools.islice(f, 3):
        print("  ", line.rstrip())

In [ ]:
# ================= PIPELINE STAGES (har ek chhota, lazy generator) =================

def read_lines(fileobj):
    # Stage 1: line by line read, khali lines chhod do
    for line in fileobj:
        line = line.strip()
        if line:                     # khali line skip
            yield line

def only_level(lines, level):
    # Stage 2: sirf wo lines jinme diya gaya level is
    for line in lines:
        if f" {level} " in line:
            yield line

def parse_record(lines):
    # Stage 3: line then dict me todo -> structured data
    for line in lines:
        parts = line.split()
        rec = {
            "date": parts[0],
            "time": parts[1],
            "level": parts[2],
        }
        # baqi hisse key=value form me are
        for token in parts[3:]:
            if "=" in token:
                k, v = token.split("=", 1)
                rec[k] = v
        yield rec

def add_seconds_field(records):
    # Stage 4: ms then float seconds me convert kar of new field lagao
    for rec in records:
        rec["seconds"] = round(int(rec["ms"]) / 1000, 3)
        yield rec

def slow_only(records, threshold=1.0):
    # Stage 5: sirf slow requests
    for rec in records:
        if rec["seconds"] >= threshold:
            yield rec

print("Stages define is gaye. Gaur do: abhi tak some bhi execute not happened (sab lazy).")

In [ ]:
# ================= PIPELINE ASSEMBLE + CONSUME =================

with open_fake_log() as f:
    # Pipeline jodo -- this ek nested generator chain is, abhi bhi some not chala
    lines    = read_lines(f)
    errors   = only_level(lines, "error")
    records  = parse_record(errors)
    enriched = add_seconds_field(records)

    print("Pipeline bana -- pehla record maangte are:")
    first = next(enriched)          # SIRF ek record of for poora chain 1 baar chala
    print("  ", first)

    print()
    print("Baqi error records:")
    for rec in enriched:            # baqi stream do are
        print(f"   user={rec['user']:<6} msg={rec.get('msg','-'):<15} {rec['seconds']}s")

In [ ]:
# ================= Pipeline from AGGREGATION (asli reporting) =================
from collections import Counter

with open_fake_log() as f:
    # Poora log -> parse -> enrich -> sirf slow (>=1s) -> count by user
    pipeline = slow_only(add_seconds_field(parse_record(read_lines(f))), threshold=1.0)
    slow_records = list(pipeline)          # chhota result, is for materialize to do theek is

print("Slow requests (>= 1.0s):")
for r in slow_records:
    print(f"   {r['time']}  {r['level']:<5} user={r['user']:<6} action={r['action']:<8} {r['seconds']}s")

print()
by_user = Counter(r["user"] for r in slow_records)
print("Slow requests per user:", dict(by_user))
assert by_user["ali"] == 2          # ali of 2 slow requests (upload 1.5s + payment 1.1s)

# One and report: error reasons
with open_fake_log() as f:
    reasons = Counter(
        r.get("msg", "unknown")
        for r in parse_record(only_level(read_lines(f), "error"))
    )
print("error reasons     :", dict(reasons))
assert reasons["card_declined"] == 2

### Output of matlab / Pipeline of faide

1. **Memory O(1)** — kabhi bhi sirf ek line + ek dict memory me tha.
2. **Composable** — `slow_only(...)` hatao, `only_level(...)` badlo — stages independent are.
3. **Testable** — har stage then alag from test kar can is (chhota input do, output check do).
4. **Early exit sasta** — `next(enriched)` ne poora log parhne of zaroorat not of.

**Interview tip:** if "10 GB log file process to do is, memory 2 GB is" that sawal aaye — jawab
**generator pipeline** is. this batao of `read → filter → parse → aggregate` sab lazy rahenge and
sirf final aggregate (Counter) memory me hoga.

---
## 12. `itertools` — Python of lazy toolkit

`itertools` stdlib module is jo **iterator building blocks** gives is. Sab C me likhe happened, fast, and
**lazy** (memory O(1)). Interview and real code dono me bohat kaam aata is.

### Three categories

**1. Infinite iterators**
| Function | Kaam |
|---|---|
| `count(start, step)` | `start, start+step, ...` hamesha |
| `cycle(iterable)` | iterable then ghoom ghoom of dohrata is |
| `repeat(obj, times=None)` | `obj` then baar baar (times no do then infinite) |

**2. Terminating (input but khatam)**
| Function | Kaam |
|---|---|
| `chain(a, b, ...)` | kai iterables then jodta is |
| `chain.from_iterable(iters)` | iterables of iterable then flatten |
| `islice(it, stop)` / `islice(it, start, stop, step)` | lazy slicing |
| `tee(it, n)` | ek iterator of `n` independent copies |
| `zip_longest(a, b, fillvalue=)` | chhoti then `fillvalue` from bharta is |
| `compress(data, selectors)` | mask of mutabiq filter |
| `dropwhile(pred, it)` | jab tak `pred` True, drop do; phir sab |
| `takewhile(pred, it)` | jab tak `pred` True, lo; first False but ruk jao |
| `filterfalse(pred, it)` | `filter` of ulta |
| `groupby(it, key)` | **consecutive** same-key items then group |
| `accumulate(it, func)` | running total / running func |
| `starmap(func, tuples)` | `func(*tup)` har tuple but |
| `pairwise(it)` | consecutive jodey: `(a,b), (b,c), ...` (3.10+) |
| `batched(it, n)` | `n` size of chunks (3.12+) |

**3. Comwithouttoric**
| Function | Kaam | Count |
|---|---|---|
| `product(a, b)` | cartesian product (nested loops) | `len(a)*len(b)` |
| `product(a, repeat=n)` | n-fold product | `len(a)**n` |
| `permutations(it, r)` | tarteeb **matters** | `n!/(n-r)!` |
| `comwithouttions(it, r)` | tarteeb **matter not**, repeat not | `C(n,r)` |
| `comwithouttions_with_replacement(it, r)` | tarteeb not, repeat allowed | `C(n+r-1, r)` |

In [ ]:
import itertools as it_

# ================= 1) INFINITE: count, cycle, repeat =================

# count(): range like magar infinite and float step bhi runs is
print("count(10)      :", list(it_.islice(it_.count(10), 5)))          # 10,11,12,13,14
print("count(0, 5)    :", list(it_.islice(it_.count(0, 5), 5)))        # 0,5,10,15,20
print("count(1, 0.5)  :", list(it_.islice(it_.count(1, 0.5), 5)))      # float step -> range not kar can

# cycle(): round-robin / repeating pattern of for perfect
colors = it_.cycle(["red", "green", "blue"])
print("cycle 7 baar   :", [next(colors) for _ in range(7)])

# repeat(): fixed value; times do then finite
print("repeat('x', 4) :", list(it_.repeat("x", 4)))
print("repeat infinite:", list(it_.islice(it_.repeat(0), 3)))

# repeat of classic use: map/zip of sath constant argument
print("map+repeat pow :", list(map(pow, [2, 3, 4, 5], it_.repeat(2))))  # sab of square

In [ ]:
# ================= 2) chain, chain.from_iterable, islice =================

a, b, c = [1, 2], (3, 4), "56"
print("chain          :", list(it_.chain(a, b, c)))            # [1,2,3,4,'5','6']

# from_iterable: jab iterables khud ek list me hon (1-level flatten)
nested = [[1, 2], [3, 4, 5], [], [6]]
print("from_iterable  :", list(it_.chain.from_iterable(nested)))
# NOTE: this sirf 1 level flatten does is. Deep flatten of for recursion (above dekha) should.

# islice: lazy slicing -- generator but bhi runs is (jahan [a:b] kaam not does)
letters = it_.count(ord("a"))
print("islice(3)      :", [chr(x) for x in it_.islice(letters, 3)])
print("islice(2,7)    :", list(it_.islice(range(20), 2, 7)))          # start,stop
print("islice(0,20,4) :", list(it_.islice(range(20), 0, 20, 4)))      # step
# GOTCHA: islice negative index support not does (because lazy is, aakhir pata not)
try:
    list(it_.islice(range(10), -3, None))
except Valueerror as e:
    print("islice(-3) ->  Valueerror:", e)

In [ ]:
# ================= 3) tee -- ek stream of kai copies =================

def expensive_stream():
    for i in range(1, 6):
        print(f"      [source] {i} generate happened")     # dikhane of for
        yield i

# tee 2 independent iterators gives is, magar source SIRF EK BAAR runs is
s1, s2 = it_.tee(expensive_stream(), 2)
print("s1 from pehli 2:", [next(s1), next(s1)])
print("s2 from pehli 2:", [next(s2), next(s2)], " <- source dobara not chala (buffer from aaya)")
print("s1 baqi      :", list(s1))
print("s2 baqi      :", list(s2))

### `tee()` of 2 ahem gotchas

1. **Original iterator then chhod do.** `tee()` of after asal iterator then use to do undefined behaviour is.
2. **Memory:** if ek copy bohat ahead nikal jaye and dusri behind reh jaye, then `tee` inside **buffer** rakhta
   is — memory O(farq). Do copies then roughly sath sath chalao. if data list me fit is jata is then
   `data = list(gen)` zyada simple and clean is.

In [ ]:
# ================= 4) zip vs zip_longest vs zip(strict=True) =================
names = ["Ali", "Sara", "Bilal", "Zain"]
marks = [90, 85, 78]                       # ek kam!

print("zip           :", list(zip(names, marks)))                 # Zain gayab (silently!)
print("zip_longest   :", list(it_.zip_longest(names, marks)))      # None from bhara
print("zip_longest f :", list(it_.zip_longest(names, marks, fillvalue=0)))

# Python 3.10+: strict=True from galti chup not rehti
try:
    list(zip(names, marks, strict=True))
except Valueerror as e:
    print("zip strict    -> Valueerror:", e)

**Interview tip:** `zip()` of default behaviour **silently truncate** to do is — this bug of bada source is.
Jab length equal honi should, `strict=True` write. Jab equal not is and missing values should,
`zip_longest` + `fillvalue` use do.

In [ ]:
# ================= 5) COMwithoutTORICS: product, permutations, comwithouttions =================

sizes  = ["S", "M"]
colors = ["red", "blue"]

# product = nested for loops of lazy version
print("product        :", list(it_.product(sizes, colors)))
# Barabar is:  [(s, c) for s in sizes for c in colors]
assert list(it_.product(sizes, colors)) == [(s, c) for s in sizes for c in colors]

# repeat= from n-fold product (like 3-digit PIN of sab combos)
print("product repeat :", ["".join(p) for p in it_.product("01", repeat=3)])

print("-" * 50)
# permutations: TARTEEB matter karti is (AB != BA)
print("perm(ABC, 2)   :", ["".join(p) for p in it_.permutations("ABC", 2)])   # 6
print("perm(ABC)      :", ["".join(p) for p in it_.permutations("ABC")])      # 3! = 6

print("-" * 50)
# comwithouttions: TARTEEB matter not (AB == BA), repeat not
print("comb(ABC, 2)   :", ["".join(p) for p in it_.comwithouttions("ABC", 2)])   # 3
# comwithouttions_with_replacement: repeat allowed (AA bhi)
print("comb_wr(ABC,2) :", ["".join(p) for p in it_.comwithouttions_with_replacement("ABC", 2)])  # 6

# Counts verify do are formula from
import math
n, r = 5, 3
assert len(list(it_.permutations(range(n), r))) == math.perm(n, r)
assert len(list(it_.comwithouttions(range(n), r))) == math.comb(n, r)
print("formula check  : perm(5,3) =", math.perm(5, 3), " comb(5,3) =", math.comb(5, 3))

In [ ]:
# ================= 6) groupby -- and uska SAB from BADA gotcha =================

data = [
    ("fruit", "apple"), ("fruit", "banana"),
    ("veg",   "carrot"),
    ("fruit", "mango"),                       # this "fruit" alag jagah is!
    ("veg",   "peas"),
]

print("without sort kiye (GALAT natija):")
for key, group in it_.groupby(data, key=lambda t: t[0]):
    print(f"   {key:<6} -> {[v for _, v in group]}")

print()
print("SORT to do of after (sahi natija):")
sorted_data = sorted(data, key=lambda t: t[0])       # SAME key from sort -- this lazmi is
for key, group in it_.groupby(sorted_data, key=lambda t: t[0]):
    print(f"   {key:<6} -> {[v for _, v in group]}")

### `groupby` of gotcha (interview me poocha jata is)

`itertools.groupby` **sirf consecutive** (lagatar) same-key items then group does is — SQL of `GROUP BY`
not is. Is for:

> **Rule: `groupby` from first usi key but `sorted()` do.**

Dusra gotcha: group ek **lazy iterator** is jo agle group but go hi **invalid** is jata is.
if after me should then foran `list(group)` bana lo.

if sirf counting/grouping should and sorting of kharcha not to do, then `collections.defaultdict(list)`
ya `Counter` zyada seedha (and O(n)) is.

In [ ]:
# ---- groupby gotcha #2: group iterator "expire" is jata is ----
groups = list(it_.groupby(sorted("mississippi")))       # (char, group) tuples of list bana li
print("keys:", [k for k, g in groups])
print("first group of values:", list(groups[0][1]), " <- KHALI! group expire is gaya")

print()
print("Sahi tareeqa -- foran list bana lo:")
runs = [(k, list(g)) for k, g in it_.groupby(sorted("mississippi"))]
for k, g in runs:
    print(f"   '{k}' x {len(g)}")

# Practical: run-length style counting -- defaultdict from zyada seedha jab data already sorted is
print("counts:", {k: len(g) for k, g in runs})
assert {k: len(g) for k, g in runs} == {"i": 4, "m": 1, "p": 2, "s": 4}

In [ ]:
# ================= 7) accumulate -- running totals =================
import operator

sales = [100, 250, 75, 300, 125]

print("running sum    :", list(it_.accumulate(sales)))                       # default = add
print("running max    :", list(it_.accumulate(sales, max)))                   # running maximum
print("running product:", list(it_.accumulate([1, 2, 3, 4, 5], operator.mul)))
print("initial= of sath:", list(it_.accumulate(sales, initial=0)))            # 0 from shuru (1 extra item)

# Real use: bank balance of running statement
opening = 5000
txns = [-1200, 3000, -450, -800]
balances = list(it_.accumulate(txns, initial=opening))
print()
print("Opening balance:", opening)
for t, bal in zip(txns, balances[1:]):
    sign = "+" if t > 0 else "-"
    print(f"   {sign}{abs(t):<6} -> balance {bal}")
assert balances[-1] == opening + sum(txns)

In [ ]:
# ================= 8) starmap, compress, filterfalse =================

# starmap: jab arguments already tuples me packed hon -> func(*tup)
pairs = [(2, 3), (4, 2), (10, 3)]
print("map    (lambda):", list(map(lambda p: pow(p[0], p[1]), pairs)))   # unpack manually
print("starmap        :", list(it_.starmap(pow, pairs)))                  # clean!

# compress: boolean mask from filter (numpy of boolean indexing like)
students = ["Ali", "Sara", "Bilal", "Zain", "Hina"]
passed   = [True, False, True, True, False]
print("compress       :", list(it_.compress(students, passed)))

# filterfalse: filter of ulta
nums = range(10)
print("filter   (even):", list(filter(lambda n: n % 2 == 0, nums)))
print("filterfalse    :", list(it_.filterfalse(lambda n: n % 2 == 0, nums)))   # odd

In [ ]:
# ================= 9) takewhile vs dropwhile (dono "first False" but switch do are) =================
readings = [2, 4, 6, 1, 8, 10, 3]

print("data           :", readings)
print("takewhile(<7)  :", list(it_.takewhile(lambda n: n < 7, readings)))
print("dropwhile(<7)  :", list(it_.dropwhile(lambda n: n < 7, readings)))
print("filter(<7)     :", list(filter(lambda n: n < 7, readings)), " <- farq gaur do!")

### `takewhile` / `dropwhile` vs `filter`

- `takewhile` **pehli** False but hamesha of for ruk jata is (baqi data dekhta bhi not).
- `dropwhile` **pehli** False tak drop does is, phir **sab** give gives is (after me shart check not does).
- `filter` **har** item then check does is.

`takewhile` + `dropwhile` = `partition at first change` — sorted data but bohat useful (and lazy, is for
infinite streams but bhi runs is).

In [ ]:
# ================= 10) pairwise (3.10+) and batched (3.12+) =================

temps = [22, 25, 24, 28, 30, 27]

# pairwise: consecutive jodey -> differences, trends, validations of for perfect
print("pairwise      :", list(it_.pairwise(temps)))
diffs = [b - a for a, b in it_.pairwise(temps)]
print("din of farq   :", diffs)
print("kitne din garm happened?", sum(1 for d in diffs if d > 0))

# is-sorted check ek line me
def is_sorted(seq):
    return all(a <= b for a, b in it_.pairwise(seq))
print("is_sorted([1,2,2,5]):", is_sorted([1, 2, 2, 5]))
print("is_sorted([1,9,2]) :", is_sorted([1, 9, 2]))

print("-" * 50)
# batched: fixed-size chunks (aakhri chunk chhota is can is)
ids = list(range(1, 11))
print("batched(3)    :", [list(b) for b in it_.batched(ids, 3)])

# Real use: API then 4-4 of batch me bhejna
for i, batch in enumerate(it_.batched(ids, 4), start=1):
    print(f"   POST /bulk  batch {i}: {list(batch)}")

# Python 3.13+ me strict=True bhi is (aakhri chunk chhota is then error)
try:
    print("batched strict:", [list(b) for b in it_.batched(ids, 3, strict=True)])
except (Valueerror, Typeerror) as e:
    print("batched strict-> ", type(e).__name__, ":", e)

---
## 13. Ek "bara file" lazily parhna (`io.StringIO`)

Real duniya me tum `open(path)` do is and `for line in f:` chalate is — **this already lazy is**.
File object khud ek iterator is! Poori file `read()` to do hi ghalti is.

```python
# ❌ 10 GB file, 2 GB RAM -> Memoryerror
data = open("huge.log").read()

# ❌ this bhi poori file memory me laata is
lines = open("huge.log").readlines()

# ✅ lazy: ek time me ek line
with open("huge.log") as f:
    for line in f:
        process(line)
```

Notebook me outside of file but depend not kar can (spec rule), is for `io.StringIO` from simulate do are
— iska interface exactly file like is.

In [ ]:
# ---- Ek "bara" CSV-like dataset banate are (memory me, magar stream of tarah parhenge) ----

def make_big_csv(rows=20_000):
    # StringIO me headelay + rows -- this humari "file on disk" is
    buf = io.StringIO()
    buf.write("ordelay_id,city,amount,status\n")
    cities = ["Lahore", "Karachi", "Islamabad", "Multan", "Quetta"]
    statuses = ["paid", "pending", "failed"]
    for i in range(rows):
        buf.write(f"{1000+i},{cities[i % 5]},{(i * 37) % 5000 + 100},{statuses[i % 3]}\n")
    buf.seek(0)                    # cursor wapis shuru me -- warna some parha not jayega
    return buf

f = make_big_csv()
print("pehli 4 lines:")
for line in it_.islice(f, 4):
    print("  ", line.rstrip())
f.close()
print("\n(gaur do: sirf 4 lines parhin, poori 20,000 not)")

In [ ]:
# ---- Lazy aggregation: 20,000 rows, memory O(1) ----
from collections import Counter

def csv_dicts(fileobj):
    # Stage 1: headelay from keys nikalo, phir har line then dict me todo
    headelay = next(fileobj).strip().split(",")     # pehli line = headelay
    for line in fileobj:
        values = line.strip().split(",")
        if len(values) == len(headelay):            # corrupt line guard
            yield dict(zip(headelay, values, strict=True))

def typed(records):
    # Stage 2: amount then int create
    for rec in records:
        rec["amount"] = int(rec["amount"])
        yield rec

t0 = time.perf_counter()
f = make_big_csv()
total_by_city = Counter()
status_count  = Counter()
row_count = 0
for rec in typed(csv_dicts(f)):        # <- streaming: ek time me ek dict
    row_count += 1
    if rec["status"] == "paid":
        total_by_city[rec["city"]] += rec["amount"]
    status_count[rec["status"]] += 1
f.close()
elapsed = time.perf_counter() - t0

print(f"rows process kiye : {row_count:,}  ({elapsed*1000:.0f} ms)")
print("status breakdown  :", dict(status_count))
print("paid revenue/city :")
for city, amt in total_by_city.most_common():
    print(f"   {city:<10} Rs {amt:>10,}")
assert row_count == 20_000

### Output of matlab

20,000 rows process happened magar memory me kabhi bhi **ek dict** from zyada not tha (plus chhote Counters).
Yehi code 20 million rows but bhi exactly isi memory me chalega — sirf time zyada lagega.

**Interview tip:** *"big file how process karoge?"* → *"File object khud iterator is; `for line in f`
from stream karoonga, generator pipeline lagaoonga, and sirf aggregates memory me rakhoonga. `read()` /
`readlines()` kabhi not."*

---
## 14. Common Mistakes / Gotchas — ek jagah sab

| # | Ghalti | what is is | Hal |
|---|---|---|---|
| 1 | Iterator then do baar loop to do | Dusri baar khali | `list()` bana lo ya new iterator |
| 2 | `len(generator)` | `Typeerror` | `sum(1 for _ in gen)` (magar consume is jayega) |
| 3 | `generator[0]` | `Typeerror` | `next(gen)` ya `islice` |
| 4 | `list(infinite_gen)` | Hang / Memoryerror | `islice` / `takewhile` / `break` |
| 5 | `groupby` without `sorted` | Toote happened groups | first usi key but sort do |
| 6 | `groupby` group then after me use to do | Khali | Foran `list(group)` |
| 7 | Generator then loop me late-binding closure of tarah samajhna | Ghalat values | Neeche demo see |
| 8 | Exception generator of inside from outside aana | Generator mar jata is | inside `try/except` |
| 9 | `return` of value expect to do | `for` ignore kar gives is | `StopIteration.value` |
| 10 | `tee` of after original use to do | Undefined behaviour | Original chhod do |
| 11 | Generator me file band to do bhool jana | Resource leak | `with` inside rakho / `close()` |
| 12 | Genexp of inside late-evaluated variable | Ghalat natija | Neeche demo see |

In [ ]:
# ---- Gotcha #12: generator LAZY is, is for outside of variable badalne from natija badal jata is ----

multiplier = 2
gen = (x * multiplier for x in range(5))     # abhi some calculate not happened

multiplier = 10                              # ab badal diya!
print("natija:", list(gen), " <- 10 from multiply happened, 2 from not!")

# HAL 1: foran materialize do
multiplier = 2
lst = [x * multiplier for x in range(5)]      # listcomp foran runs is
multiplier = 10
print("listcomp:", lst, " <- 2 from hi happened (safe)")

# HAL 2: value then default argument me "freeze" kar do
def make_gen(m):
    return (x * m for x in range(5))          # m function of scope me qaid is gaya
multiplier = 2
g2 = make_gen(multiplier)
multiplier = 10
print("closure :", list(g2), " <- 2 from happened (safe)")

In [ ]:
# ---- Gotcha #7: generator expression of PEHLA `for` foran evaluate is is (baqi lazy) ----

data = [1, 2, 3]
gen = (x for x in data)      # `data` of reference ABHI liya gaya

data = [100, 200]            # naam rebind done -> generator but asar not
print("rebind of after:", list(gen), " <- purani list from aaya")

data2 = [1, 2, 3]
gen2 = (x for x in data2)
data2.append(4)              # SAME object mutate done -> generator but asar HOGA
print("mutate of after:", list(gen2), " <- 4 bhi aa gaya!")

In [ ]:
# ---- Gotcha #11: generator of inside file/resource -- `with` inside rakho ----

def bad_readelay(text):
    f = io.StringIO(text)
    for line in f:
        yield line.strip()
    # if consumer beech me break kar give then f.close() kabhi not chalega!

def good_readelay(text):
    with io.StringIO(text) as f:      # `with` generator of inside -> close guaranteed
        for line in f:
            yield line.strip()
    # generator close/GC hone but `with` of __exit__ runs is

sample = "line1\nline2\nline3\nline4"

# Consumer beech me break does is
g = good_readelay(sample)
for i, line in enumerate(g):
    if i == 1:
        break
print("beech me break done. generator close do are:")
g.close()                  # -> `with` of cleanup chal jayega
print("cleanup is gaya (koi resource leak not)")

print("counting without consuming? -- not is can:")
g2 = good_readelay(sample)
print("   count:", sum(1 for _ in g2))     # count get gaya
print("   ab list:", list(g2), " <- khali, because count ne consume kar liya")

---
## 15. Real-World problems

Chalo ab 4 asli masle hal do are — wahi cheezein jo kaam but gets are.

### problem 1 — big file me duplicate records dhoondo (memory O(unique))

**Scenario:** 5 GB transaction log is. Duplicate `txn_id` dhoondne are magar poori file RAM me not aa can.

**Approach:** Stream do, ek `set` me dekhe happened IDs rakho. Memory sirf unique IDs of barabar,
poori file of barabar not. (if unique IDs bhi bohat zyada hon then Bloom filter / external sort should.)

In [ ]:
def find_duplicate_ids(lines):
    # Lazy generator: like hi duplicate mile, foran yield kar do (poori file of infastar not)
    seen = set()                       # sirf IDs, poore records not
    for line in lines:
        txn_id = line.split(",", 1)[0]  # sirf pehla field parse do (baqi of zaroorat not)
        if txn_id in seen:
            yield txn_id                # duplicate got
        else:
            seen.add(txn_id)

txn_log = io.StringIO('''\
T100,ali,500
T101,sara,300
T102,bilal,900
T101,sara,300
T103,zain,150
T100,ali,500
T104,hina,700
''')

dups = list(find_duplicate_ids(txn_log))
print("duplicate txn ids:", dups)
assert dups == ["T101", "T100"]

# Pehla duplicate get hi ruk jana (early exit) -- lazy hone of faida
txn_log2 = io.StringIO("A,1\nB,2\no,3\nC,4\nB,5\n")
print("pehla duplicate  :", next(find_duplicate_ids(txn_log2)))

### problem 2 — Infinite retry schedule (exponential backoff)

**Scenario:** API call fail is then dobara try do — magar har baar zyada infastar do (0.5s, 1s, 2s, 4s...)
and maximum cap lagao. Delay schedule ek **infinite generator** is; caller jitne should le le.

**Approach:** `itertools.count()` from attempt number, `min(base * 2**n, cap)` from delay. Generator hone of
wajah from this schedule reusable, testable, and retry-logic from alag (decoupled) is.

In [ ]:
def backoff_delays(base=0.5, cap=30.0, jitter=0.0):
    # Infinite generator: attempt 0, 1, 2, ... of for delay
    for attempt in it_.count():
        delay = min(base * (2 ** attempt), cap)      # exponential, magar cap from zyada not
        yield round(delay + jitter, 3)

# first 8 attempts of schedule see (koi sleep not -- sirf numbers)
print("schedule:", list(it_.islice(backoff_delays(), 8)))
assert list(it_.islice(backoff_delays(base=1, cap=10), 6)) == [1, 2, 4, 8, 10, 10]

print()
# Ab isko ek fake flaky API of sath use do are (real sleep bohat chhota rakha is)
call_count = {"n": 0}
def flaky_api():
    call_count["n"] += 1
    if call_count["n"] < 4:                    # pehli 3 calls fail
        raise Connectionerror(f"attempt {call_count['n']} fail")
    return {"status": "ok", "attempt": call_count["n"]}

def call_with_retry(fn, max_attempts=6):
    # delays generator from get are -- retry logic clean reh gaya
    delays = backoff_delays(base=0.001, cap=0.01)     # test of for micro-delays
    for attempt, delay in enumerate(it_.islice(delays, max_attempts), start=1):
        try:
            return fn()
        except Connectionerror as e:
            print(f"   attempt {attempt} fail ({e}) -> {delay}s after retry")
            time.sleep(delay)
    raise Runtimeerror("saare attempts fail is gaye")

result = call_with_retry(flaky_api)
print("result:", result)
assert result["attempt"] == 4

### problem 3 — Sliding window over a stream (moving average)

**Scenario:** Sensor from readings aa rahi are (stream). Har naye reading but pichhle `k` of average should.
Poora data store not kar can.

**Approach:** `collections.deque(maxlen=k)` — jab new item aata is, old khud nikal jata is.
Generator hone of wajah from this infinite stream but bhi runs is, memory O(k).

In [ ]:
from collections import deque

def moving_average(stream, k):
    window = deque(maxlen=k)          # maxlen = old khud gir jata is (O(1))
    total = 0.0
    for value in stream:
        if len(window) == k:
            total -= window[0]        # jo gir jayega usko total from ghatao
        window.append(value)
        total += value
        # sirf tab yield do jab poora window bhar jaye
        if len(window) == k:
            yield round(total / k, 2)

readings = [10, 12, 23, 23, 16, 3, 15, 20]
avgs = list(moving_average(iter(readings), 3))
print("readings:", readings)
print("MA(3)   :", avgs)
assert avgs == [15.0, 19.33, 20.67, 14.0, 11.33, 12.67]

# Infinite stream but bhi runs is -- memory sirf 5 items
noisy_stream = (abs((i * 7919) % 100) for i in it_.count())
print("infinite stream of MA(5), first 6:", list(it_.islice(moving_average(noisy_stream, 5), 6)))

### problem 4 — Do sorted streams then merge to do (external merge sort of core)

**Scenario:** Do big sorted files are. Unko ek sorted stream me merge to do is — without dono then RAM me laane of.

**Approach:** Dono of `next()` compare do, chhota that yield do. Memory O(1).
Stdlib me this already is: `heapq.merge()` — and wo N streams handle does is.

In [ ]:
import heapq

def merge_two_sorted(a, b):
    # Manual implementation -- protocol samajhne of for
    ia, ib = iter(a), iter(b)
    x = next(ia, None)                     # sentinel None = stream khatam
    y = next(ib, None)
    while x is not None and y is not None:
        if x <= y:
            yield x
            x = next(ia, None)
        else:
            yield y
            y = next(ib, None)
    # jo bacha happened is wo seedha give do
    while x is not None:
        yield x
        x = next(ia, None)
    while y is not None:
        yield y
        y = next(ib, None)

s1 = [1, 4, 7, 10, 13]
s2 = [2, 3, 8, 20]
merged = list(merge_two_sorted(s1, s2))
print("stream A:", s1)
print("stream B:", s2)
print("merged  :", merged)
assert merged == sorted(s1 + s2)

# Stdlib of tareeqa: heapq.merge -- N streams, lazy, memory O(N)
s3 = [0, 5, 9]
print("heapq.merge (3 streams):", list(heapq.merge(s1, s2, s3)))
assert list(heapq.merge(s1, s2, s3)) == sorted(s1 + s2 + s3)

# key= bhi support does is -- magar har input stream USI key from sorted hona should
words = [["fig", "apple"], ["kiwi", "banana"]]      # dono length of hisab from sorted are
print("key=len from merge:", list(heapq.merge(*words, key=len)))
assert list(heapq.merge(*words, key=len)) == ["fig", "kiwi", "apple", "banana"]

---
# 16. LeetCode problems

this section iterator/generator protocol then **interview problems** but lagata is. Char problems —
teen "design an iterator" type (jahan `__next__` khud likhna is) and ek bonus jo `itertools` from
elegantly hal is is.

### LC 341. Flatten Nested List Iterator — Medium

**Link:** https://leetcode.com/problems/flatten-nested-list-iterator/

**Bayan (English):** Tumhe ek nested list di gayi is, like `[[1,1],2,[1,1]]`. Har element ya then ek
**integer** is ya khud ek **list** (jo and nested is can is). Ek iterator class create jiske `next()` and
`hasNext()` methods flattened ordelay me integers dein.

LeetCode `NestedInteger` interface gives is jiske paas `isInteger()`, `getInteger()`, `getList()` are —
hum wo class khud inline bana lenge taake code standalone chale.

**Approach 1 — Eager flatten (easy, samajhne me clean):**
1. Constructor me recursion from **poori** nested list then ek flat list me kholo.
2. Ek index pointer rakho. `hasNext()` = `index < len(flat)`. `next()` = value do, index++.
- Trade-off: constructor O(N) time **and** O(N) memory — poora data first hi flat is jata is.

**Approach 2 — Lazy stack (LeetCode of "asli" jawab):**
1. Stack me list of elements **ulta** daalo (taake top but pehla element is).
2. `hasNext()` me: jab tak stack of top ek **list** is, use pop do and uske elements ulta push do.
   Jab top integer ban jaye then `True`.
3. `next()` first `hasNext()` chalata is (normalize), phir `stack.pop()`.
- Faida: constructor O(1)-ish, memory O(depth + breadth) — poora flatten not is.

**Approach 3 — Generator (Pythonic):** `yield from` from recursive generator — code sab from chhota.

### Complexity

| Approach | Constructor | `next()` | `hasNext()` | Space |
|---|---|---|---|---|
| Eager flatten | O(N) | O(1) | O(1) | O(N) |
| Lazy stack | O(top-level) | **amortized O(1)** | amortized O(1) | O(N) worst (deep nesting) |
| Generator | O(1) | amortized O(1) | O(1) (peek buffer) | O(depth) |

`N` = total integers. Amortized O(1) because har element stack but sirf ek baar push/pop is is.

In [ ]:
# ================= LC 341: Flatten Nested List Iterator =================

class NestedInteger:
    # LeetCode of interface -- hum khud bana rahe are taake standalone chale.
    def __init__(self, value=None):
        # value int is then integer node, list is then list node
        if isinstance(value, list):
            self._list = [v if isinstance(v, NestedInteger) else NestedInteger(v) for v in value]
            self._int = None
        else:
            self._list = None
            self._int = value

    def isInteger(self):
        return self._int is not None

    def getInteger(self):
        return self._int

    def getList(self):
        return self._list

    def __repr__(self):
        return repr(self._int) if self.isInteger() else repr(self._list)


def build(raw):
    # Python list-of-(int|list) then NestedInteger objects of list me badalta is
    return [NestedInteger(x) for x in raw]


# ---------- Approach 1: EAGER flatten ----------
class NestedIteratorEager:
    def __init__(self, nestedList):
        self._flat = []
        self._flatten(nestedList)       # constructor me hi poora khol diya
        self._i = 0

    def _flatten(self, items):
        for ni in items:
            if ni.isInteger():
                self._flat.append(ni.getInteger())
            else:
                self._flatten(ni.getList())      # recursion -- deeper level

    def hasNext(self):
        return self._i < len(self._flat)

    def next(self):
        val = self._flat[self._i]
        self._i += 1
        return val


# ---------- Approach 2: LAZY stack (optimal) ----------
class NestedIterator:
    def __init__(self, nestedList):
        # reversed() taake list of PEHLA element stack of TOP but aa jaye
        self._stack = list(reversed(nestedList))

    def hasNext(self):
        # Top then "normalize" do: jab tak top list is, usko kholte jao
        while self._stack:
            top = self._stack[-1]
            if top.isInteger():
                return True                       # top ready is
            self._stack.pop()                     # list thi -> nikaalo
            self._stack.extend(reversed(top.getList()))   # uske children ulta push
        return False                              # stack khali -> khatam

    def next(self):
        # hasNext() call to do zaroori is (LeetCode guarantee gives is magar safe rehte are)
        if not self.hasNext():
            raise StopIteration("koi element baqi not")
        return self._stack.pop().getInteger()

    # Bonus: Python iterator protocol bhi give dete are -> `for x in itr` chalega
    def __iter__(self):
        return self

    def __next__(self):
        if not self.hasNext():
            raise StopIteration
        return self._stack.pop().getInteger()


# ---------- Approach 3: GENERATOR (Pythonic) ----------
def flatten_nested(nestedList):
    # yield from from recursion -- 5 line me kaam
    for ni in nestedList:
        if ni.isInteger():
            yield ni.getInteger()
        else:
            yield from flatten_nested(ni.getList())


def drain(itr):
    # hasNext/next those iterator then list me convert does is (testing helper)
    out = []
    while itr.hasNext():
        out.append(itr.next())
    return out


# ================= TESTS =================
t1 = [[1, 1], 2, [1, 1]]
t2 = [1, [4, [6]]]
t3 = []                                  # khali
t4 = [[]]                                # sirf khali lists -> koi integer not
t5 = [[], [1], [], [2, [3, [4, []]]], 5]  # deep + khali mix

assert drain(NestedIteratorEager(build(t1))) == [1, 1, 2, 1, 1]
assert drain(NestedIterator(build(t1)))      == [1, 1, 2, 1, 1]
assert list(flatten_nested(build(t1)))       == [1, 1, 2, 1, 1]

assert drain(NestedIterator(build(t2))) == [1, 4, 6]
assert drain(NestedIterator(build(t3))) == []
assert drain(NestedIterator(build(t4))) == []          # edge case: khali lists
assert drain(NestedIterator(build(t5))) == [1, 2, 3, 4, 5]

# Teeno approaches of jawab hamesha same hona should
for raw in (t1, t2, t3, t4, t5):
    a = drain(NestedIteratorEager(build(raw)))
    b = drain(NestedIterator(build(raw)))
    c = list(flatten_nested(build(raw)))
    assert a == b == c, raw

print("LC 341 of saare tests PASS")
print("  [[1,1],2,[1,1]] ->", drain(NestedIterator(build(t1))))
print("  [1,[4,[6]]]     ->", drain(NestedIterator(build(t2))))
print("  deep+khali mix  ->", drain(NestedIterator(build(t5))))
print("  for-loop (bonus)->", [x for x in NestedIterator(build(t2))])

### LC 284. Peeking Iterator — Medium

**Link:** https://leetcode.com/problems/peeking-iterator/

**Bayan (English):** Ek normal iterator then wrap do and usme `peek()` method add do jo **agli value
dikhaye magar iterator then ahead no barhaye**. `next()` and `hasNext()` normal kaam do rahein.

**Approach — one-element lookahead buffer:**
1. inside ek `_buffer` rakho jisme "agli value" first from nikaal of rakhi is, and ek flag `_has_buffer`.
2. `peek()`: if buffer khali is then undelaylying from ek value nikaal of buffer me daal do; buffer return do.
3. `next()`: if buffer bhara is then wahi return do and buffer khali kar do; warna undelaylying from lo.
4. `hasNext()`: `buffer bhara is` **ya** undelaylying me some baqi is.

**Ahem baat:** undelaylying iterator me "some baqi is?" poochne of koi tareeqa not is — pata lagane of
sirf ek zariya is: `next()` maar of see. Isi for lookahead buffer should. Isi then **eager fetch** kehte are.

**Gotcha:** `peek()` of side-effect from undelaylying **already** ahead barh jata is. if undelaylying koi
expensive/one-shot stream is (network) then `peek()` of matlab is "ek record first hi fetch is gaya".

### Complexity
- `peek()`, `next()`, `hasNext()`: **O(1)** (amortized, sirf ek buffered element)
- Space: **O(1)** — chahe stream kitni bhi big is

In [ ]:
# ================= LC 284: Peeking Iterator =================

class PeekingIterator:
    def __init__(self, iterator):
        # iterator = koi bhi Python iterator (list_iterator, generator, map object...)
        self._it = iter(iterator)
        self._buffer = None            # ek value of lookahead
        self._has_buffer = False       # buffer valid is ya not (None bhi valid value is can is)
        self._fill()                   # constructor me hi ek value pre-fetch (hasNext of for lazmi)

    def _fill(self):
        # Undelaylying from ek value nikaal of buffer me daalne of koshish
        if not self._has_buffer:
            try:
                self._buffer = next(self._it)
                self._has_buffer = True
            except StopIteration:
                self._has_buffer = False    # stream khatam

    def peek(self):
        # Agli value dikhao MAGAR ahead no barho
        if not self._has_buffer:
            raise StopIteration("peek: koi element baqi not")
        return self._buffer

    def next(self):
        # Buffer that value do, phir agli pre-fetch do
        if not self._has_buffer:
            raise StopIteration("next: koi element baqi not")
        val = self._buffer
        self._has_buffer = False
        self._fill()                   # agla element buffer me
        return val

    def hasNext(self):
        return self._has_buffer

    # Python-friendly banate are
    def __iter__(self):
        return self

    def __next__(self):
        if not self._has_buffer:
            raise StopIteration
        return self.next()


# ================= TESTS =================
# LeetCode of example: nums = [1,2,3]
p = PeekingIterator(iter([1, 2, 3]))
assert p.next() == 1          # 1 got, pointer 2 but
assert p.peek() == 2          # 2 dikha magar pointer wahin
assert p.peek() == 2          # dobara peek -> phir 2 (idempotent)
assert p.next() == 2          # ab 2 consume happened
assert p.next() == 3
assert p.hasNext() is False

# Edge case: khali iterator
e = PeekingIterator(iter([]))
assert e.hasNext() is False
try:
    e.peek()
    raise Assertionerror("yahan StopIteration aana should tha")
except StopIteration:
    pass

# Generator but bhi runs is (sirf iterator protocol should)
p2 = PeekingIterator(x * x for x in range(1, 5))     # 1,4,9,16
seen = []
while p2.hasNext():
    seen.append((p2.peek(), p2.next()))
assert seen == [(1, 1), (4, 4), (9, 9), (16, 16)]

# Infinite stream but bhi -- because space O(1) is
p3 = PeekingIterator(it_.count(100))
assert p3.peek() == 100 and p3.next() == 100 and p3.peek() == 101

# Practical use: peek from "next group" decide to do (without consume kiye)
def take_while_peek(pit, pred):
    out = []
    while pit.hasNext() and pred(pit.peek()):    # PEEK but decision, phir consume
        out.append(pit.next())
    return out

p4 = PeekingIterator(iter([1, 2, 3, 10, 11, 4]))
print("LC 284 of saare tests PASS")
print("  <5 those prefix :", take_while_peek(p4, lambda x: x < 5))
print("  baqi           :", [x for x in p4])

### LC 173. withoutry Search Tree Iterator — Medium

**Link:** https://leetcode.com/problems/withoutry-search-tree-iterator/

**Bayan (English):** Ek BST diya gaya is. Aisa iterator create jo `next()` but **in-ordelay** (yani
**sorted**) tarteeb from agla chhota element give, and `hasNext()` bataye of and element baqi is ya not.
Shart: `next()` and `hasNext()` **average O(1)** time me hon and memory **O(h)** is (h = tree of height) —
yani poora tree flatten kar of list me rakhna "cheating" is.

**Approach 1 — Eager in-ordelay (easy):** poora in-ordelay traversal kar of list bana lo + index.
Space O(N) — LeetCode of follow-up shart tordta is, magar samajhne of for acha starting point.

**Approach 2 — Controlled recursion / stack (optimal):**
1. Constructor: root from shuru kar of **saare left children** stack but push do.
2. `next()`: stack from pop do — **yehi** sab from chhota baqi node is. Us node of **right** child but jao
   and us from neeche of saare left children push do. Node of value return do.
3. `hasNext()`: `len(stack) > 0`.

why kaam does is? Stack me hamesha "wo ancestors jinki value abhi visit not happened" are are — yani
in-ordelay traversal then haath from pause/resume to do.

**Approach 3 — Generator (Pythonic):** in-ordelay generator + `PeekingIterator` like lookahead. Recursion
of state Python khud manage kar takes is — **is for code sab from clean is**. Interview me first stack that
batao (because wo language-agnostic is), phir "Python me main generator use does" add do.

### Complexity

| Approach | `next()` | `hasNext()` | Space |
|---|---|---|---|
| Eager list | O(1) | O(1) | O(N) ❌ |
| Stack | **amortized O(1)** (worst O(h)) | O(1) | **O(h)** ✅ |
| Generator | amortized O(1) | O(1) | O(h) ✅ |

Amortized O(1) because poore traversal me har node exactly ek baar push and ek baar pop is is → N
operations / N calls = O(1) per call.

In [ ]:
# ================= LC 173: withoutry Search Tree Iterator =================

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right


def build_bst(values):
    # Values then ek ek kar of BST me insert does is (test trees banane of for)
    root = None
    for v in values:
        node = TreeNode(v)
        if root is None:
            root = node
            continue
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = node
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = node
                    break
                cur = cur.right
    return root


# ---------- Approach 1: EAGER in-ordelay ----------
class BSTIteratorEager:
    def __init__(self, root):
        self._vals = []
        self._inordelay(root)        # poora tree ek list me (space O(N))
        self._i = 0

    def _inordelay(self, node):
        if node is None:
            return
        self._inordelay(node.left)   # left
        self._vals.append(node.val)  # khud
        self._inordelay(node.right)  # right

    def next(self):
        v = self._vals[self._i]
        self._i += 1
        return v

    def hasNext(self):
        return self._i < len(self._vals)


# ---------- Approach 2: STACK (optimal, space O(h)) ----------
class BSTIterator:
    def __init__(self, root):
        self._stack = []
        self._push_left_spine(root)     # sab from chhote tak of raasta stack but

    def _push_left_spine(self, node):
        # node from shuru kar of jitne left children are sab push
        while node:
            self._stack.append(node)
            node = node.left

    def next(self):
        # Stack of top = agla sab from chhota (in-ordelay successor)
        node = self._stack.pop()
        # Its right subtree of left-spine ab stack but aana should
        if node.right:
            self._push_left_spine(node.right)
        return node.val

    def hasNext(self):
        return len(self._stack) > 0


# ---------- Approach 3: GENERATOR (Pythonic) ----------
def bst_inordelay(node):
    # Lazy in-ordelay generator -- recursion of state Python sambhalta is
    if node is None:
        return
    yield from bst_inordelay(node.left)
    yield node.val
    yield from bst_inordelay(node.right)


class BSTIteratorGen:
    def __init__(self, root):
        # Generator then PeekingIterator style buffer of sath wrap done
        self._gen = bst_inordelay(root)
        self._buf = None
        self._has = False
        self._advance()

    def _advance(self):
        try:
            self._buf = next(self._gen)
            self._has = True
        except StopIteration:
            self._has = False

    def next(self):
        v = self._buf
        self._advance()
        return v

    def hasNext(self):
        return self._has


# ================= TESTS =================
def drain_bst(itr):
    out = []
    while itr.hasNext():
        out.append(itr.next())
    return out

# LeetCode example: root = [7,3,15,null,null,9,20]
root = TreeNode(7, TreeNode(3), TreeNode(15, TreeNode(9), TreeNode(20)))

itr = BSTIterator(root)
assert itr.next() == 3        # LeetCode of expected sequence
assert itr.next() == 7
assert itr.hasNext() is True
assert itr.next() == 9
assert itr.hasNext() is True
assert itr.next() == 15
assert itr.hasNext() is True
assert itr.next() == 20
assert itr.hasNext() is False

# Teeno approaches: jawab sorted ordelay hona should
for vals in ([7, 3, 15, 9, 20], [50], [], [5, 3, 8, 1, 4, 7, 9], [1, 2, 3, 4, 5]):
    r = build_bst(vals)
    a = drain_bst(BSTIteratorEager(r))
    b = drain_bst(BSTIterator(r))
    c = drain_bst(BSTIteratorGen(r))
    assert a == b == c == sorted(vals), vals

print("LC 173 of saare tests PASS")
print("  tree [7,3,15,null,null,9,20] ->", drain_bst(BSTIterator(root)))
print("  skewed tree (1..5)           ->", drain_bst(BSTIterator(build_bst([1, 2, 3, 4, 5]))))
print("  khali tree                   ->", drain_bst(BSTIterator(build_bst([]))))

### LC 1291. Sequential Digits — Medium *(bonus — `itertools` of elegant use)*

**Link:** https://leetcode.com/problems/sequential-digits/

**Bayan (English):** "Sequential digits" wo number is jiske har digit apne pichhle digit from **exactly 1
zyada** is (like `123`, `4567`, `89`). `[low, high]` range me aane those saare aise numbers **sorted** ordelay
me return do.

**Approach — brute force but sochna chhod do, generate do:**
Aise numbers total **sirf 36** are (1 from 9 tak of digits, length 2 from 9 tak) — because start digit and
length dono limited are. then saare generate do and filter do.

1. String `"123456789"` but har possible **contiguous slice** (length 2..9) lo.
2. Usko `int` create.
3. `low <= n <= high` those rakho, `sorted()` kar do.

`itertools.comwithouttions` of zaroorat not — simple nested slicing kaafi is, and `itertools.chain` from
generation lazy rakh can is.

**why this "iterator" notebook me?** because this us soch of misaal is: *"answer space chhota is → generate
do (lazily), search no do"*. this generator-first thinking is.

### Complexity
- Time: **O(1)** — candidates fixed 36 are (`sorted` but bhi O(36 log 36)). Input size but depend not.
- Space: **O(1)** (output of ilawa)

In [ ]:
# ================= LC 1291: Sequential Digits (bonus) =================

DIGITS = "123456789"

def sequential_candidates():
    # Lazy generator: chhoti length first -> natija roughly sorted aata is
    for length in range(2, 10):                     # length 2 from 9 tak
        for start in range(0, 10 - length):          # kahan from slice shuru is
            yield int(DIGITS[start:start + length])

def sequentialDigits(low, high):
    # Generator pipeline: generate -> filter -> sort
    return sorted(n for n in sequential_candidates() if low <= n <= high)

# Total candidates gin lete are -- sirf 36
all_cands = list(sequential_candidates())
print("total sequential numbers:", len(all_cands))
print("first 12                :", all_cands[:12])
assert len(all_cands) == 36

# ================= TESTS (LeetCode examples) =================
assert sequentialDigits(100, 300) == [123, 234]
assert sequentialDigits(1000, 13000) == [1234, 2345, 3456, 4567, 5678, 6789, 12345]
assert sequentialDigits(10, 100) == [12, 23, 34, 45, 56, 67, 78, 89]
assert sequentialDigits(1, 9) == []                 # single digit count not is
assert sequentialDigits(123, 123) == [123]          # exact match
assert sequentialDigits(123456789, 123456789) == [123456789]   # sab from bara

print("LC 1291 of saare tests PASS")
print("  (100, 300)      ->", sequentialDigits(100, 300))
print("  (1000, 13000)   ->", sequentialDigits(1000, 13000))

---
# 17. Interview questions & answers

**Q1. Iterable and iterator me farq?**
Iterable of paas `__iter__()` is is — us from iterator banew ja can is (`list`, `str`, `dict`).
Iterator of paas `__iter__()` **and** `__next__()` dono are are, wo apni position of state rakhta is,
and **one-shot** is is. Har iterator iterable is; har iterable iterator not.

**Q2. `StopIteration` exception is — phir `for` loop crash why not is?**
`for` loop inside from `StopIteration` then pakadta is and usko "loop khatam" of signal samajh kar `break`
kar gives is. this control-flow signal is, error not.

**Q3. Generator and normal function me farq?**
Normal function `return` but ek dafa value give kar **khatam** is jata is (local state gayab). Generator
function me `yield` is is — call to do but body chalti not, ek generator object milta is; har `next()`
but body `yield` tak chal kar **suspend** is jati is and locals mehfooz rehte are.

**Q4. Generator and list me what farq, and kab kaun sa?**
List poora data memory me rakhti is (O(N)), reusable is, indexing/`len()` support karti is.
Generator lazy is (O(1) memory), one-shot is, indexing/`len()` not. Bara/infinite/streaming data →
generator. Chhota data + reuse/indexing → list.

**Q5. `yield` vs `return` generator me?**
`yield` value gives is and function suspend does is. `return` generator then **band** kar gives is; uski
value `StopIteration.value` me chali jati is (jo `for` loop ignore does is).

**Q6. `yield from` of faida sirf syntax is?**
not. Chhote code of ilawa wo `send()`, `throw()`, and sub-generator of `return` value then **properly
forward** does is. Manual `for x in sub: yield x` this sab not does.

**Q7. Generator expression and list comprehension me farq?**
Brackets: `()` vs `[]`. Listcomp foran poori list banati is; genexp ek lazy iterator gives is.
`sum(x*x for x in big)` memory-efficient is, `sum([x*x for x in big])` not.

**Q8. Ek generator then do baar loop how do?**
not kar can. Options: (a) `data = list(gen)` bana lo, (b) generator **function** then dobara call do
(new generator banega), (c) `itertools.tee()` (magar buffering memory takes is).

**Q9. `itertools.groupby` and SQL `GROUP BY` me farq?**
`groupby` sirf **consecutive** same-key items group does is. Is for usi key but `sorted()` to do zaroori
is. Warna ek hi key of kai toote happened groups milenge.

**Q10. `next(gen)` and `gen.send(None)` me farq?**
Koi farq not — dono generator then ahead barhate are and `yield` expression of value `None` set do are.
`send(x)` from `x` value inject is is, magar first generator **primed** hona should.

**Q11. `gen.close()` what does is?**
Generator of current `yield` but `GeneratorExit` raise does is. Generator use pakad kar cleanup kar can
is (`try/finally` ya `except GeneratorExit`). if generator `GeneratorExit` of after `yield` kare then
`Runtimeerror` aata is.

**Q12. Generator me exception handle no is then what is is?**
Exception caller tak propagate is is and generator **band** is jata is — dobara `next()` but
`StopIteration` milega. Resilient generator of for loop of inside `try/except` lagao.

**Q13. `zip()`, `map()`, `filter()` Python 3 me list dete are ya iterator?**
Iterator. Is for do baar loop not is and `len()` not runs. Python 2 me this lists dete the — this
Py2→Py3 of classic bug source is.

**Q14. `itertools.islice` and normal slicing `[a:b]` me farq?**
`islice` **kisi bhi** iterator but runs is and lazy is, magar negative indices support not does
(because aakhir kahan is this pata not). Normal slicing sirf sequences but runs is, negative allow is,
and new object banata is.

**Q15. `tee()` of memory cost what is?**
`O(largest gap)` — if ek copy N items ahead nikal jaye then `tee` inside N items buffer does is. if dono
copies then poora ghumana is then `list(gen)` hi behtar and clean is.

**Q16. Lazy evaluation of nuksan what is?**
(a) Debugging mushkil — error consume do time aata is, define do time not.
(b) Do baar traverse not. (c) `len()` not. (d) if outside of mutable state badal jaye then natija badal
can is (deferred evaluation trap). (e) Resource (file) generator of inside khula reh can is.

**Q17. "Generator pipeline" of faida ek line me?**
Memory O(1) + composable stages + early exit sasta. 50 GB file 100 MB RAM me process is jati is.

**Q18. `iter()` of do-argument form what is?**
`iter(callable, sentinel)` — `callable()` then baar baar call does is jab tak wo `sentinel` return no kare.
File then fixed-size chunks me parhne of classic idiom: `iter(lambda: f.read(4096), b"")`.

**Q19. Generator thread-safe is?**
not. Ek generator then kai threads from `next()` to do race condition is (`Valueerror: generator already
executing` bhi get can is). Har thread then apna generator do, ya lock lagao, ya `queue.Queue` use do.

**Q20. Generator function me `yield` is magar wo kabhi no chale — phir bhi generator is?**
yes. `yield` of **maujoodgi** compile-time but function then generator bana deti is, chahe wo line kabhi
execute no is. Is for `def f(): return; yield` ek khali generator is, `None` return to do that function not.

In [ ]:
# ---- Q18 of demo: iter() of do-argument (sentinel) form ----

# Misaal 1: file then fixed chunks me parhna (classic idiom)
f = io.StringIO("A" * 25)                  # 25 characters of "file"
chunks = list(iter(lambda: f.read(10), ""))  # jab read("") give -> ruk jao
print("chunks:", [len(c) for c in chunks], "->", chunks)
assert [len(c) for c in chunks] == [10, 10, 5]

# Misaal 2: fake queue from "poll" to do jab tak sentinel no mile
queue_data = iter([3, 7, 9, None, 11])      # None = "bas"
polled = list(iter(lambda: next(queue_data), None))
print("sentinel from first:", polled)
assert polled == [3, 7, 9]

# ---- Q20 of demo: yield of maujoodgi function then generator bana deti is ----
def never_yields():
    return                    # foran return
    yield                     # this line kabhi not chalti -- magar function generator is!

r = never_yields()
print("type:", type(r).__name__, "| values:", list(r))
import inspect
assert inspect.isgeneratorfunction(never_yields)
print("inspect.isgeneratorfunction:", inspect.isgeneratorfunction(never_yields))

---
# 18. Practice Exercises (khud do)

Har exercise of for function write and apne `assert` from test do. **Hint** phanse but see.

**1. `take(n, iterable)`** — kisi bhi iterable from pehli `n` values of list do (if kam are then jitni are).
   *Hint:* `itertools.islice`.

**2. `chunked(iterable, size)`** — generator jo lists of chunks give: `chunked(range(7), 3)` →
   `[0,1,2], [3,4,5], [6]`. `list()` me poora data no aaye.
   *Hint:* `itertools.islice` then `while True` loop me chalao and khali chunk but `break`, ya `batched`
   from compare do.

**3. `unique(iterable)`** — ordelay barqarar rakhte happened duplicates hatao, **lazy** generator create.
   Bonus: `key=` parameter add do (like `key=str.lower`).
   *Hint:* `seen = set()`; unhashable items of for what karoge?

**4. `windowed(iterable, n)`** — sliding windows: `windowed([1,2,3,4], 2)` → `(1,2), (2,3), (3,4)`.
   *Hint:* `deque(maxlen=n)`; `n=2` but `itertools.pairwise` from apna jawab verify do.

**5. `interleave(*iterables)`** — baari baari ek ek item lo: `interleave([1,2,3], "abc")` →
   `1, 'a', 2, 'b', 3, 'c'`. Unequal lengths handle do (chhoti khatam is jaye then usko chhod do).
   *Hint:* `zip_longest` + sentinel, ya `cycle` of iterators.

**6. `count_lines_matching(fileobj, pattern)`** — without poori file memory me laane of un lines of ginti do
   jinme `pattern` is. `io.StringIO` from test do.
   *Hint:* `sum(1 for line in f if pattern in line)`.

**7. `primes()`** — infinite generator jo prime numbers give. `islice` from first 20 nikalo and verify do.
   *Hint:* trial division kaafi is; optimize: sirf `sqrt(n)` tak, and mile happened primes from divide do.

**8. `flatten_iter(nested, depth=None)`** — recursive flatten magar `depth` limit of sath.
   `depth=1` sirf ek level khole. Strings then atomic understand (warna infinite recursion!).
   *Hint:* `isinstance(x, str)` of special case first check do.

**9. `batched_api_sendelay(records, batch_size)`** — records then batches me "bhejo" (print do) and
   har batch of summary dict yield do: `{"batch": 1, "count": 4}`.
   *Hint:* `itertools.batched`.

**10. `TopN` iterator class** — stream from guzarte happened top-N sab from big values remember, memory O(N).
   `__iter__` sorted descending ordelay give.
   *Hint:* `heapq` (min-heap of size N) — ya `heapq.nlargest` from apna jawab check do.

**11. (Challenge) `merge_sorted(*streams, key=None)`** — N sorted streams then merge do, lazy, memory O(N).
   `heapq.merge` from apna jawab compare do — magar khud implement do.
   *Hint:* min-heap me `(key(value), stream_index, value, iterator)` tuples rakho.

**12. (Challenge) `Peekable` class** — `PeekingIterator` of general version jo `peek(n)` support kare
   (agli `n` values dikhaye without consume kiye).
   *Hint:* buffer `deque` create and zaroorat of mutabiq `_fill(n)` do.

In [ ]:
# ---- Self-check helper: apne jawab yahan test do ----
# Below exercise 1 and 3 of solutions are -- baqi khud do!

def take(n, iterable):
    # All from clean tareeqa: islice lazy is and kisi bhi iterable but runs is
    return list(it_.islice(iterable, n))

assert take(3, range(10)) == [0, 1, 2]
assert take(5, [1, 2]) == [1, 2]              # kam items -> jitne are
assert take(0, range(10)) == []               # edge case
assert take(3, it_.count()) == [0, 1, 2]      # infinite but bhi runs is
print("exercise 1 (take) PASS ->", take(4, "PYTHON"))


def unique(iterable, key=None):
    # Lazy dedupe, ordelay preserve
    seen = set()
    for item in iterable:
        k = item if key is None else key(item)
        if k not in seen:
            seen.add(k)
            yield item                        # pehli baar got -> give do

assert list(unique([1, 3, 1, 2, 3, 4, 2])) == [1, 3, 2, 4]
assert list(unique("mississippi")) == ["m", "i", "s", "p"]
assert list(unique(["A", "a", "B", "b"], key=str.lower)) == ["A", "B"]
# lazy hone of saboot: infinite stream but bhi runs is
assert take(4, unique(it_.cycle([5, 5, 6, 7, 6, 8]))) == [5, 6, 7, 8]
print("exercise 3 (unique) PASS ->", list(unique([1, 3, 1, 2, 3, 4, 2])))

print()
print("Baqi 10 exercises khud do -- above of hints read!")

---
# 19. Summary / Cheat-sheet

## Protocol ek nazar me

```
iterable  --iter()-->  iterator  --next()-->  value ... StopIteration
```
- **Iterable** = `__iter__`  ·  **Iterator** = `__iter__` + `__next__`
- Iterator **one-shot** is. `for` loop = `iter()` + `next()` loop + `StopIteration` but `break`.

## Iterator banane of 3 tareeqe

| Tareeqa | Code | Kab |
|---|---|---|
| Class | `__iter__` + `__next__` + `raise StopIteration` | Complex state, reusable iterable should |
| Generator function | `def g(): yield x` | **99% cases** — sab from clean |
| Generator expression | `(f(x) for x in it)` | One-liner transformation/filter |

## Generator of methods

| Call | Kaam |
|---|---|
| `next(g)` / `g.send(None)` | ahead barhao |
| `g.send(v)` | `yield` of jagah `v` inject do (**first prime do!**) |
| `g.throw(Exc)` | inside exception raise do |
| `g.close()` | `GeneratorExit` → cleanup |
| `yield from sub` | Delegation (send/throw/return sab forward) |

## `itertools` cheat-sheet

```python
# Infinite (islice/takewhile from kaato!)
count(10, 2)          # 10, 12, 14, ...
cycle("abc")          # a, b, c, a, b, c, ...
repeat("x", 3)        # x, x, x

# Jodna / kaatna
chain([1,2], [3])              # 1, 2, 3
chain.from_iterable([[1],[2]]) # 1, 2      (1-level flatten)
islice(it, 2, 9, 2)            # lazy slicing (negative index not)
tee(it, 2)                     # 2 copies (buffer memory!)
zip_longest(a, b, fillvalue=0) # chhoti then bharo

# Filter / transform
compress(data, [1,0,1])        # mask from filter
filterfalse(pred, it)          # filter of ulta
takewhile(pred, it)            # pehli False but RUK jao
dropwhile(pred, it)            # pehli False tak drop, phir SAB
starmap(pow, [(2,3)])          # func(*tuple)
accumulate(nums)               # running sum  (func=, initial= bhi)
groupby(SORTED_it, key)        # consecutive groups -- SORT LAZMI

# Windows / chunks
pairwise([1,2,3])              # (1,2), (2,3)          3.10+
batched(range(7), 3)           # (0,1,2), (3,4,5), (6,) 3.12+

# Comwithouttorics
product(a, b)                  # nested loops    len(a)*len(b)
product("01", repeat=3)        # n-fold          len**n
permutations("abc", 2)         # tarteeb MATTERS  n!/(n-r)!
comwithouttions("abc", 2)         # tarteeb not     C(n,r)
comwithouttions_with_replacement("abc", 2)   # repeat allowed
```

## Lazy vs Eager decision

| should | Use |
|---|---|
| `len()`, indexing, sorting, dobara loop | **list** |
| Bara/infinite data, streaming, sirf ek traversal | **generator** |
| Pehli matching value | `next(x for x in it if cond)` |
| Poora aggregate (`sum`/`max`/`Counter`) | genexp + aggregate function |

## Top 8 gotchas

1. Iterator do baar loop → dusri baar **khali**
2. `zip`/`map`/`filter`/`enumerate` = iterators (Py3)
3. `len(gen)` / `gen[0]` → `Typeerror`
4. `list(infinite_gen)` → hang
5. `groupby` without `sorted` → toote groups; group **expire** is jata is
6. Genexp lazy is → outside of variable badal gaya then natija badal jayega
7. `tee` of after original chhod do; gap = memory
8. Generator of inside file? `with` **inside** rakho, warna leak

## LeetCode is notebook from

| LC | Title | Difficulty | Core idea |
|---|---|---|---|
| 341 | Flatten Nested List Iterator | Medium | Lazy stack / `yield from` recursion |
| 284 | Peeking Iterator | Medium | One-element lookahead buffer |
| 173 | withoutry Search Tree Iterator | Medium | Left-spine stack, amortized O(1), space O(h) |
| 1291 | Sequential Digits | Medium | answer space chhota → generate, search not |

## Interview one-liners

- *"`for` loop how runs is?"* → `iter()` → `next()` loop → `StopIteration` but break.
- *"Generator why?"* → Lazy: memory O(1), pehli value foran, infinite sequences mumkin.
- *"10 GB file?"* → Generator pipeline; sirf aggregates memory me.
- *"`yield from` why?"* → Recursion clean + send/throw/return forwarding.

---
### ahead what?
Agla notebook: **02 — Closures, Decorators and Functional Python** (`functools`, `yield` of bad ab
higher-ordelay functions).